# CNN Predict — CESM2-LE JJA SST → September Slowdown

Loads **precomputed CNN predictions** and reproduces the same figures as `cnn_train.ipynb` **without re-training or re-running inference**. Also generates composite maps of the SST input and LRP attributions for TP / FP / TN / FN scenarios.

**Prerequisites**
- `python scripts/04_cesm2le_cnn_train.py` — trains models and saves JSON histories
- `python scripts/05_cesm2le_lrp.py` — computes and saves LRP attributions
- `python scripts/06_cnn_predict_cesm2le.py` — precomputes and saves predictions

**Set `N_SPLITS` and `N_SEEDS` to match the values used during training, then Run All.**

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths
from src.cnn.splits import load_tvt_split
from src.cnn.model  import METRIC_NAMES

# ── Publication figure defaults ───────────────────────────────────────────────
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})

## Config
Must match the values used in `cnn_train.ipynb` (or `scripts/04_cesm2le_cnn_train.py`).

In [ ]:
# ── How many splits / seeds were trained ─────────────────────────────────────
N_SPLITS  = 9          # max 9
N_SEEDS   = 5          # random seeds per split
BASE_SEED = 42

# ── Dataset ───────────────────────────────────────────────────────────────────
BLOCK_SIZE = 10        # ensemble members per TVT block
START_YEAR = 1990      # first year in the loaded splits

METRICS_DIR     = paths.RESULTS_DIR / 'metrics'
PREDICTIONS_DIR = paths.RESULTS_DIR / 'predictions' / 'cesm2le'

## Load loop
Loads TVT splits (for labels and SST arrays), **precomputed predictions**, metrics, training histories, and LRP attributions.
Results are stored in `results[split_idx]`.

> **Note:** Model inference is no longer performed here. Predictions are loaded from
> files produced by `scripts/06_cnn_predict_cesm2le.py`.

In [ ]:
# results[split_idx] = {
#   'histories':     list[dict]        — one per seed (loaded from JSON)
#   'y_scores_runs': list[dict]        — one per seed, keys: train/val/test
#   'y_true':        dict              — keys: train/val/test
#   'ds_metrics':    xr.Dataset
#   'lrp_data':      list[dict|None]   — one per seed; None if LRP file missing
#   'sst_tr':        np.ndarray        — (n_tr, nx, ny) training SST
#   'lat':           np.ndarray        — (nlat,) 1-D latitude from LRP file
#   'lon':           np.ndarray        — (nlon,) 1-D longitude from LRP file
# }
results = {}

for split_idx in range(N_SPLITS):
    print(f'\n{"─"*60}')
    print(f'Split {split_idx}')
    print(f'{"─"*60}')

    # ── TVT split (for labels and SST arrays) ─────────────────────────────────
    split_path = paths.tvt_split_path(split_idx)
    if not split_path.exists():
        raise FileNotFoundError(
            f'TVT split not found: {split_path}\n'
            f'Run scripts/03_cesm2le_tvt_splits.py first.'
        )
    split = load_tvt_split(split_path)

    y_tr = split['slow_tr']
    y_va = split['slow_va']
    y_te = split['slow_te']
    y_true = {'train': y_tr, 'val': y_va, 'test': y_te}

    print(f'  Train: ({len(y_tr)},)  prevalence={y_tr.mean():.3f}')
    print(f'  Val  : ({len(y_va)},)  prevalence={y_va.mean():.3f}')
    print(f'  Test : ({len(y_te)},)  prevalence={y_te.mean():.3f}')

    histories      = []
    y_scores_runs  = []
    lrp_data       = []
    lat_shared     = None
    lon_shared     = None

    for run_idx in range(N_SEEDS):
        print(f'  seed {BASE_SEED + run_idx}: ', end='', flush=True)

        # ── Load precomputed predictions ──────────────────────────────────
        pred_path = PREDICTIONS_DIR / f'cnn_prediction_cesm2le_M{split_idx}_{run_idx}.nc'
        if not pred_path.exists():
            raise FileNotFoundError(
                f'Prediction file not found: {pred_path}\n'
                f'Run scripts/06_cnn_predict_cesm2le.py first.'
            )
        with xr.open_dataset(pred_path) as ds_pred:
            y_scores_runs.append({
                'train': ds_pred['y_prob_train'].values,
                'val':   ds_pred['y_prob_val'].values,
                'test':  ds_pred['y_prob_test'].values,
            })
        print('predictions ✓', end='  ', flush=True)

        # ── Training history ──────────────────────────────────────────────
        hist_path = paths.LOGS_DIR / f'history_split{split_idx}_run{run_idx}.json'
        if hist_path.exists():
            with open(hist_path) as f:
                histories.append(json.load(f))
        else:
            histories.append(None)
            print(f'(history not found: {hist_path})', end='  ')
        print('history ✓', end='  ', flush=True)

        # ── LRP attributions ──────────────────────────────────────────────
        lrp_path = paths.attribution_path(split_idx, run_idx)
        if lrp_path.exists():
            lrp_ds   = xr.open_dataset(lrp_path)
            lrp_raw  = lrp_ds['lrp_attributions'].values   # (n_chunks, chunk_size, nx, ny, 1)
            lat_1d   = lrp_ds['lat'].values                 # (nx,)
            lon_1d   = lrp_ds['lon'].values                 # (ny,)
            lrp_ds.close()
            # flatten to (n_lrp, nx, ny)
            lrp_flat = lrp_raw.reshape(-1, lrp_raw.shape[2], lrp_raw.shape[3])
            lrp_data.append({'lrp_flat': lrp_flat})
            if lat_shared is None:
                lat_shared = lat_1d
                lon_shared = lon_1d
            print('LRP ✓')
        else:
            lrp_data.append(None)
            print(f'(LRP not found: {lrp_path})')

    # ── Metrics dataset ───────────────────────────────────────────────────────
    metrics_p = paths.metrics_path(split_idx)
    if not metrics_p.exists():
        raise FileNotFoundError(
            f'Metrics not found: {metrics_p}\n'
            f'Run scripts/04_cesm2le_cnn_train.py first.'
        )
    ds_metrics = xr.open_dataset(metrics_p)

    results[split_idx] = {
        'histories':     histories,
        'y_scores_runs': y_scores_runs,
        'y_true':        y_true,
        'ds_metrics':    ds_metrics,
        'lrp_data':      lrp_data,
        'sst_tr':        split['sst_tr'],   # (n_tr, nx, ny) — no channel dim
        'lat':           lat_shared,
        'lon':           lon_shared,
    }

print('\nLoading complete.')

---
## Visualization

**Plots 0–2 and 4** show results for a single split × seed — set `PLOT_SPLIT` and `PLOT_SEED` below.  
**Plot 3** aggregates across all loaded splits and seeds.

In [5]:
PLOT_SPLIT = 0    # which split to visualize (0 to N_SPLITS-1)
PLOT_SEED  = 0    # which seed to visualize  (0 to N_SEEDS-1)

# Convenience references
res      = results[PLOT_SPLIT]
hist     = res['histories'][PLOT_SEED]    # dict loaded from JSON (or None)
y_scores = res['y_scores_runs'][PLOT_SEED]
y_true   = res['y_true']

### 0 — Learning curves

In [ ]:
if hist is None:
    print(f'No history file for split {PLOT_SPLIT}, seed {PLOT_SEED}.')
else:
    epochs = np.arange(1, len(hist['loss']) + 1)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, hist['loss'],     color='steelblue', linewidth=2.0, label='train loss')
    ax.plot(epochs, hist['val_loss'], color='steelblue', linewidth=2.0, label='val loss',
            linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('(a)')
    ax.legend()
    plt.tight_layout()
    plt.show()

### 1 — Precision–Recall curve & threshold selection

In [ ]:
y_s = y_scores['test']
y_t = y_true['test']

precision, recall, thresholds = precision_recall_curve(y_t, y_s)

# F1 at each threshold
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)

# Threshold at precision–recall intersection (same rule as compute_metrics)
best_idx = np.argmin(np.abs(precision[:-1] - recall[:-1]))
best_thr = thresholds[best_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precision[:-1], label='Precision', color='steelblue', linewidth=2.0)
ax.plot(thresholds, recall[:-1],    label='Recall',    color='darkorange', linewidth=2.0)
ax.plot(thresholds, f1_scores,      label='F1',        color='seagreen', linewidth=2.0)
ax.axvline(best_thr, color='black', linewidth=2, label=f'threshold = {best_thr:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('(a)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Selected threshold: {best_thr:.4f}  '
      f'(Precision={precision[best_idx]:.3f}, Recall={recall[best_idx]:.3f})')

### 2 — Confusion matrices (train + test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for idx, (ax, split_name) in enumerate(zip(axes, ['train', 'test'])):
    yt = y_true[split_name]
    ys = y_scores[split_name]

    prec, rec, thr = precision_recall_curve(yt, ys)
    if thr.size > 0:
        thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))]
    else:
        thr_best = 0.5

    y_pred = (ys >= thr_best).astype(int)
    cm     = confusion_matrix(yt, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Negative', 'Positive']
    )
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    label = chr(ord('a') + idx)
    ax.set_title(f'({label})')

plt.tight_layout()
plt.show()

### 3 — Evaluation metrics across all seeds & splits
Mean value per metric on the test split, with individual seed/split values overlaid.

In [ ]:
all_vals = []
for sidx in range(N_SPLITS):
    ds = results[sidx]['ds_metrics']
    all_vals.append(ds['metric_value'].sel(split='test').values)  # (n_metrics, n_seeds)

all_vals = np.stack(all_vals, axis=0)               # (n_splits, n_metrics, n_seeds)
all_vals = all_vals.transpose(1, 0, 2)              # (n_metrics, n_splits, n_seeds)

DISPLAY_METRICS = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUPRC', 'AUROC', 'MCC']
display_idx     = [METRIC_NAMES.index(m) for m in DISPLAY_METRICS]

vals_display = all_vals[display_idx, :, :]
vals_flat    = vals_display.reshape(len(DISPLAY_METRICS), -1)
means        = np.nanmean(vals_flat, axis=1)

fig, ax = plt.subplots(figsize=(9, 5))

for i, (m, mean_val) in enumerate(zip(DISPLAY_METRICS, means)):
    jitter = np.random.uniform(-0.15, 0.15, size=vals_flat.shape[1])
    ax.scatter(
        np.full(vals_flat.shape[1], i) + jitter,
        vals_flat[i],
        color='steelblue', alpha=0.5, s=25, zorder=2
    )
    ax.plot([i - 0.3, i + 0.3], [mean_val, mean_val],
            color='darkred', linewidth=3.0, zorder=3)

ax.set_xticks(range(len(DISPLAY_METRICS)))
ax.set_xticklabels(DISPLAY_METRICS)
ax.set_ylabel('Score (%)')
ax.set_title('(a)')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}'))
plt.tight_layout()
plt.show()

### 4 — Slowdown time series per ensemble member
Shows actual slowdowns (gray), correct predictions (green), and incorrect predictions (red) for each test-block member.

In [ ]:
yt = y_true['test']
ys = y_scores['test']

prec, rec, thr = precision_recall_curve(yt, ys)
thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
y_pred   = (ys >= thr_best).astype(int)

nyears   = yt.shape[0] // BLOCK_SIZE
years    = np.arange(START_YEAR, START_YEAR + nyears)
yt_2d    = yt.reshape(BLOCK_SIZE, nyears)
ypred_2d = y_pred.reshape(BLOCK_SIZE, nyears)

test_block_start = PLOT_SPLIT * BLOCK_SIZE
member_labels    = np.arange(test_block_start, test_block_start + BLOCK_SIZE)

fig, ax = plt.subplots(figsize=(16, 5))

for m_idx, m_label in enumerate(member_labels):
    actual    = yt_2d[m_idx]
    predicted = ypred_2d[m_idx]

    ax.axhline(m_label, color='lightgray', linewidth=0.6, zorder=0)

    actual_yrs = years[actual == 1]
    ax.scatter(actual_yrs, np.full_like(actual_yrs, m_label, dtype=float),
               color='dimgray', s=70, zorder=2,
               label='actual slowdowns' if m_idx == 0 else '')

    tp_yrs = years[(predicted == 1) & (actual == 1)]
    ax.scatter(tp_yrs, np.full_like(tp_yrs, m_label, dtype=float),
               color='mediumseagreen', s=30, zorder=3,
               label='correct predictions' if m_idx == 0 else '')

    err_yrs = years[(predicted == 1) & (actual == 0)]
    ax.scatter(err_yrs, np.full_like(err_yrs, m_label, dtype=float),
               color='lightcoral', s=30, zorder=3,
               label='incorrect predictions' if m_idx == 0 else '')

ax.set_xlabel('Year')
ax.set_ylabel('Ensemble member no.')
ax.set_xlim(years[0] - 1, years[-1] + 1)
ax.set_yticks(member_labels)
ax.set_title('(a)')
ax.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.12),
          frameon=False, markerscale=1.4)
plt.tight_layout()
plt.show()

---
## Composite Maps

For each selected scenario (TP / FP / TN / FN):
- **(a)** Mean JJA SST input over all scenario samples (land masked to NaN)
- **(b)** Mean LRP attribution over all scenario samples

Composites are computed per split × seed first, then averaged across all splits × seeds.

> **Note:** LRP must have been computed with `scripts/05_cesm2le_lrp.py`. Splits/seeds missing LRP files are automatically skipped.

In [11]:
# ── Toggle scenarios to plot ──────────────────────────────────────────────────
SHOW_TP = True
SHOW_FP = False
SHOW_TN = False
SHOW_FN = False

In [ ]:
SCENARIO_FLAGS = {'TP': SHOW_TP, 'FP': SHOW_FP, 'TN': SHOW_TN, 'FN': SHOW_FN}
ACTIVE_SCENARIOS = [k for k, v in SCENARIO_FLAGS.items() if v]

for scenario_name in ACTIVE_SCENARIOS:

    sst_per_sr = []
    lrp_per_sr = []
    lat_plot   = None
    lon_plot   = None

    for split_idx in range(N_SPLITS):
        res    = results[split_idx]
        sst_tr = res['sst_tr']   # (n_tr, nx, ny)

        for run_idx in range(N_SEEDS):
            lrp_info = res['lrp_data'][run_idx]
            if lrp_info is None:
                continue   # LRP not available for this split/seed

            y_s = res['y_scores_runs'][run_idx]['train']
            y_t = res['y_true']['train']

            # ── threshold ────────────────────────────────────────────────────
            prec, rec, thr = precision_recall_curve(y_t, y_s)
            thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
            y_pred   = (y_s >= thr_best).astype(int)

            # ── scenario mask ─────────────────────────────────────────────────
            if scenario_name == 'TP':
                mask = (y_pred == 1) & (y_t == 1)
            elif scenario_name == 'FP':
                mask = (y_pred == 1) & (y_t == 0)
            elif scenario_name == 'TN':
                mask = (y_pred == 0) & (y_t == 0)
            else:   # FN
                mask = (y_pred == 0) & (y_t == 1)

            # ── align LRP with training data (LRP may have fewer samples) ────
            lrp_flat = lrp_info['lrp_flat']   # (n_lrp, nx, ny)
            n_lrp    = lrp_flat.shape[0]
            mask_lrp = mask[:n_lrp]
            sst_lrp  = sst_tr[:n_lrp]         # (n_lrp, nx, ny)

            if mask_lrp.sum() == 0:
                continue

            # ── SST composite — land fill (-10) → NaN ────────────────────────
            sst_masked = sst_lrp.astype(float).copy()
            sst_masked[sst_masked < -5] = np.nan
            sst_mean = np.nanmean(sst_masked[mask_lrp], axis=0)   # (nx, ny)

            # ── LRP composite ────────────────────────────────────────────────
            lrp_mean = np.mean(lrp_flat[mask_lrp], axis=0)        # (nx, ny)

            sst_per_sr.append(sst_mean)
            lrp_per_sr.append(lrp_mean)

            if lat_plot is None:
                lat_plot = res['lat']
                lon_plot = res['lon']

    if len(sst_per_sr) == 0:
        print(f'{scenario_name}: no samples found across any split/seed — skipping.')
        continue

    n_sr      = len(sst_per_sr)
    sst_grand = np.nanmean(np.stack(sst_per_sr, axis=0), axis=0)  # (nx, ny)
    lrp_grand = np.nanmean(np.stack(lrp_per_sr, axis=0), axis=0)  # (nx, ny)

    print(f'{scenario_name}: averaged over {n_sr} split×seed composites')

    # ── Normalize ensemble-mean maps (97th percentile) ────────────────────────
    sst_scale = np.nanpercentile(np.abs(sst_grand), 97)
    sst_grand = np.clip(sst_grand / (sst_scale + 1e-12), -1, 1)

    lrp_scale = np.nanpercentile(np.abs(lrp_grand), 97)
    lrp_grand = np.clip(lrp_grand / (lrp_scale + 1e-12), -1, 1)

    # ── Build 2-D lat/lon grid for plotting ───────────────────────────────────
    lon2d, lat2d = np.meshgrid(lon_plot, lat_plot)   # (nlat, nlon)

    # ── Figure ────────────────────────────────────────────────────────────────
    proj  = ccrs.PlateCarree(central_longitude=180)

    fig = plt.figure(figsize=(14, 5))
    ax_sst = fig.add_subplot(1, 2, 1, projection=proj)
    ax_lrp = fig.add_subplot(1, 2, 2, projection=proj)

    for ax, data, vmin, vmax, cmap, label_txt, panel_lbl in [
        (ax_sst, sst_grand, -1, 1, cmocean.cm.balance, 'SST (normalized)', '(a)'),
        (ax_lrp, lrp_grand, -1, 1, cmocean.cm.curl,    'Relevance (normalized)', '(b)'),
    ]:
        ax.set_global()
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

        im = ax.pcolormesh(
            lon2d, lat2d, data,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            transform=ccrs.PlateCarree(),
            shading='auto',
            zorder=0,
        )
        plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.04,
                     fraction=0.046, label=label_txt)
        ax.set_title(panel_lbl)

    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------------
# plotting options
# ------------------------------------------------------------------
LRP_POSITIVE_ONLY = True   # False = signed relevance, True = keep only positive
LRP_POSITIVE_CMAP = 'magma'
LRP_SIGNED_CMAP   = cmocean.cm.curl

SCENARIO_FLAGS = {'TP': SHOW_TP, 'FP': SHOW_FP, 'TN': SHOW_TN, 'FN': SHOW_FN}
ACTIVE_SCENARIOS = [k for k, v in SCENARIO_FLAGS.items() if v]

for scenario_name in ACTIVE_SCENARIOS:

    sst_per_sr = []
    lrp_per_sr = []
    lat_plot   = None
    lon_plot   = None

    for split_idx in range(N_SPLITS):
        res    = results[split_idx]
        sst_tr = res['sst_tr']   # (n_tr, nx, ny)

        for run_idx in range(N_SEEDS):
            lrp_info = res['lrp_data'][run_idx]
            if lrp_info is None:
                continue

            y_s = res['y_scores_runs'][run_idx]['train']
            y_t = res['y_true']['train']

            # ── threshold ──────────────────────────────────────────────
            prec, rec, thr = precision_recall_curve(y_t, y_s)
            thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
            y_pred   = (y_s >= thr_best).astype(int)

            # ── scenario mask ──────────────────────────────────────────
            if scenario_name == 'TP':
                mask = (y_pred == 1) & (y_t == 1)
            elif scenario_name == 'FP':
                mask = (y_pred == 1) & (y_t == 0)
            elif scenario_name == 'TN':
                mask = (y_pred == 0) & (y_t == 0)
            else:   # FN
                mask = (y_pred == 0) & (y_t == 1)

            # ── align LRP with training data ───────────────────────────
            lrp_flat = lrp_info['lrp_flat']   # (n_lrp, nx, ny)
            n_lrp    = lrp_flat.shape[0]
            mask_lrp = mask[:n_lrp]
            sst_lrp  = sst_tr[:n_lrp]

            if mask_lrp.sum() == 0:
                continue

            # ── SST composite — land fill (-10) → NaN ─────────────────
            sst_masked = sst_lrp.astype(float).copy()
            sst_masked[sst_masked < -5] = np.nan
            sst_mean = np.nanmean(sst_masked[mask_lrp], axis=0)

            # ── LRP composite ──────────────────────────────────────────
            lrp_selected = lrp_flat[mask_lrp]

            if LRP_POSITIVE_ONLY:
                lrp_selected = np.where(lrp_selected > 0, lrp_selected, 0.0)

            lrp_mean = np.mean(lrp_selected, axis=0)

            sst_per_sr.append(sst_mean)
            lrp_per_sr.append(lrp_mean)

            if lat_plot is None:
                lat_plot = res['lat']
                lon_plot = res['lon']

    if len(sst_per_sr) == 0:
        print(f'{scenario_name}: no samples found across any split/seed — skipping.')
        continue

    n_sr      = len(sst_per_sr)
    sst_grand = np.nanmean(np.stack(sst_per_sr, axis=0), axis=0)
    lrp_grand = np.nanmean(np.stack(lrp_per_sr, axis=0), axis=0)

    print(f'{scenario_name}: averaged over {n_sr} split×seed composites')

    # ── Normalize ensemble-mean maps (97th percentile) ────────────────
    sst_scale = np.nanpercentile(np.abs(sst_grand), 97)
    sst_grand = np.clip(sst_grand / (sst_scale + 1e-12), -1, 1)

    if LRP_POSITIVE_ONLY:
        lrp_scale = np.nanpercentile(lrp_grand, 97)
        lrp_grand = np.clip(lrp_grand / (lrp_scale + 1e-12), 0, 1)
    else:
        lrp_scale = np.nanpercentile(np.abs(lrp_grand), 97)
        lrp_grand = np.clip(lrp_grand / (lrp_scale + 1e-12), -1, 1)

    # ── Build 2-D lat/lon grid for plotting ───────────────────────────
    lon2d, lat2d = np.meshgrid(lon_plot, lat_plot)

    # ── Figure ────────────────────────────────────────────────────────
    proj = ccrs.PlateCarree(central_longitude=180)

    fig = plt.figure(figsize=(14, 5))
    ax_sst = fig.add_subplot(1, 2, 1, projection=proj)
    ax_lrp = fig.add_subplot(1, 2, 2, projection=proj)

    # SST
    ax_sst.set_global()
    ax_sst.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax_sst.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
    ax_sst.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    im_sst = ax_sst.pcolormesh(
        lon2d, lat2d, sst_grand,
        cmap=cmocean.cm.balance,
        vmin=-1, vmax=1,
        transform=ccrs.PlateCarree(),
        shading='auto',
        zorder=0,
    )
    plt.colorbar(
        im_sst, ax=ax_sst, orientation='horizontal', pad=0.04,
        fraction=0.046, label='SST (normalized)'
    )
    ax_sst.set_title('(a)')

    # LRP
    ax_lrp.set_global()
    ax_lrp.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax_lrp.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
    ax_lrp.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    lrp_cmap = LRP_POSITIVE_CMAP if LRP_POSITIVE_ONLY else LRP_SIGNED_CMAP
    lrp_vmin = 0 if LRP_POSITIVE_ONLY else -1
    lrp_vmax = 1

    im_lrp = ax_lrp.pcolormesh(
        lon2d, lat2d, lrp_grand,
        cmap=lrp_cmap,
        vmin=lrp_vmin, vmax=lrp_vmax,
        transform=ccrs.PlateCarree(),
        shading='auto',
        zorder=0,
    )
    plt.colorbar(
        im_lrp, ax=ax_lrp, orientation='horizontal', pad=0.04,
        fraction=0.046, label='Relevance (normalized)'
    )
    ax_lrp.set_title('(b)')

    plt.tight_layout()
    plt.show()

---
### 5 — 4-panel SST composites: all events vs. CNN-filtered

Compares the raw SST composite for **all** labelled events against the
CNN-filtered composite (TP / TN) using only the CESM2 training data.

- Top row  — slowdown cases:    (a) all slowdowns        (b) TP slowdowns
- Bottom row — non-slowdown cases: (c) all non-slowdowns (d) TN non-slowdowns

Composites are built per split × seed first (truth-only panels use the
training-SST/LRP alignment so they draw on the same subset of samples used in
the TP/TN panels) and then averaged across all splits × seeds — the same
convention used by the composite-map cells above.

In [ ]:
# =============================================================================
# 4-panel SST composite figure — raw label vs. CNN-filtered prediction
#
# (a) ALL slowdowns         — y_t == 1                    (training data)
# (b) TP slowdowns          — y_pred == 1 & y_t == 1      (training data)
# (c) ALL non-slowdowns     — y_t == 0                    (training data)
# (d) TN non-slowdowns      — y_pred == 0 & y_t == 0      (training data)
# =============================================================================

composites = {'ALL_SLOW': [], 'TP': [], 'ALL_NONSLOW': [], 'TN': []}
lat_plot, lon_plot = None, None

for split_idx in range(N_SPLITS):
    res    = results[split_idx]
    sst_tr = res['sst_tr']   # (n_tr, nx, ny)

    for run_idx in range(N_SEEDS):
        lrp_info = res['lrp_data'][run_idx]
        if lrp_info is None:
            continue

        y_s = res['y_scores_runs'][run_idx]['train']
        y_t = res['y_true']['train']

        prec, rec, thr = precision_recall_curve(y_t, y_s)
        thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
        y_pred   = (y_s >= thr_best).astype(int)

        n_lrp   = lrp_info['lrp_flat'].shape[0]
        sst_lrp = sst_tr[:n_lrp]
        y_t_lrp    = y_t[:n_lrp]
        y_pred_lrp = y_pred[:n_lrp]

        masks = {
            'ALL_SLOW':    (y_t_lrp == 1),
            'TP':          (y_pred_lrp == 1) & (y_t_lrp == 1),
            'ALL_NONSLOW': (y_t_lrp == 0),
            'TN':          (y_pred_lrp == 0) & (y_t_lrp == 0),
        }

        sst_masked = sst_lrp.astype(float).copy()
        sst_masked[sst_masked < -5] = np.nan

        for key, m in masks.items():
            if m.sum() == 0:
                continue
            composites[key].append(np.nanmean(sst_masked[m], axis=0))

        if lat_plot is None:
            lat_plot = res['lat']
            lon_plot = res['lon']

# Grand composites
grand = {k: np.nanmean(np.stack(v, axis=0), axis=0) for k, v in composites.items()}
n_sr  = {k: len(v) for k, v in composites.items()}
print('Composites averaged per panel:', n_sr)

# Normalize: shared 97th percentile across all 4 panels
sst_scale = max(
    np.nanpercentile(np.abs(grand['ALL_SLOW']),    97),
    np.nanpercentile(np.abs(grand['TP']),          97),
    np.nanpercentile(np.abs(grand['ALL_NONSLOW']), 97),
    np.nanpercentile(np.abs(grand['TN']),          97),
)
for k in grand:
    grand[k] = np.clip(grand[k] / (sst_scale + 1e-12), -1, 1)

# ── Figure ────────────────────────────────────────────────────────────────────
lon2d, lat2d = np.meshgrid(lon_plot, lat_plot)
proj = ccrs.PlateCarree(central_longitude=180)

fig, axes = plt.subplots(
    2, 2, figsize=(14, 8), subplot_kw={'projection': proj}
)

panels = [
    (axes[0, 0], grand['ALL_SLOW'],    '(a)'),
    (axes[0, 1], grand['TP'],          '(b)'),
    (axes[1, 0], grand['ALL_NONSLOW'], '(c)'),
    (axes[1, 1], grand['TN'],          '(d)'),
]

for ax, data, title in panels:
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    im = ax.pcolormesh(
        lon2d, lat2d, data,
        cmap=cmocean.cm.balance,
        vmin=-1, vmax=1,
        transform=ccrs.PlateCarree(),
        shading='auto',
        zorder=0,
    )
    ax.set_title(title)

# Shared horizontal colorbar
cbar = fig.colorbar(
    im, ax=axes.ravel().tolist(), orientation='horizontal',
    fraction=0.04, pad=0.06, shrink=0.7,
)
cbar.set_label('SST (normalized)')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Rectangle

# =============================================================================
# Helper functions
# =============================================================================

def bootstrap_mean_ci(vals, B=1000, alpha=0.05, random_state=None):
    """Bootstrap 95 % CI for the mean.  Returns (mean, lower, upper)."""
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = vals.size
    if n == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    means = np.empty(B)
    for b in range(B):
        means[b] = rng.choice(vals, size=n, replace=True).mean()
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return vals.mean(), lo, hi


def bootstrap_p_value_diff_means(vals1, vals2, B=5000, random_state=None):
    """Two-sided permutation p-value for difference in means."""
    v1 = np.asarray(vals1, dtype=float);  v1 = v1[np.isfinite(v1)]
    v2 = np.asarray(vals2, dtype=float);  v2 = v2[np.isfinite(v2)]
    n1, n2 = v1.size, v2.size
    if n1 == 0 or n2 == 0:
        return np.nan
    diff_obs = v1.mean() - v2.mean()
    pooled = np.concatenate([v1, v2])
    rng = np.random.default_rng(random_state)
    diffs = np.empty(B)
    for b in range(B):
        s = rng.choice(pooled, size=pooled.size, replace=True)
        diffs[b] = s[:n1].mean() - s[n1:].mean()
    return float(np.mean(np.abs(diffs) >= np.abs(diff_obs)))


# =============================================================================
# Region definitions
# =============================================================================

REGIONS = {
    'Arctic':                  dict(lat=(60, 90),   lon=(0, 360)),
    'Tropical Pacific (ENSO)': dict(lat=(-10, 10),  lon=(170, 270)),
    'North Pacific (PDO)':     dict(lat=(15, 55),   lon=(140, 230)),
    'Pacific (IPO)':           dict(lat=(-55, 55),  lon=(140, 270)),
    'Atlantic (AMO)':          dict(lat=(0, 60),    lon=(280, 350)),
    'Indian Ocean (IOD)':      dict(lat=(-35, 15),  lon=(50, 100)),
    'Niño 1+2':               dict(lat=(-10, 0),   lon=(270, 280)),
    'Niño 3':                  dict(lat=(-5, 5),    lon=(210, 270)),
    'Niño 4':                  dict(lat=(-5, 5),    lon=(160, 210)),
}

_cmap_tab10 = plt.get_cmap('tab10')
REGION_COLORS = {name: _cmap_tab10(i % 10) for i, name in enumerate(REGIONS)}


def _region_mask(lat1d, lon1d, lat_range, lon_range):
    """Boolean mask (nlat, nlon) selecting grid cells inside a box."""
    lat2 = lat1d[:, None] if lat1d.ndim == 1 else lat1d
    lon2 = lon1d[None, :] if lon1d.ndim == 1 else lon1d
    return ((lat2 >= lat_range[0]) & (lat2 <= lat_range[1]) &
            (lon2 >= lon_range[0]) & (lon2 <= lon_range[1]))


def regional_mean_relevance(lrp_2d, lat1d, lon1d, regions):
    """Mean positive-only relevance in each region (ignoring NaN / land)."""
    out = {}
    for name, box in regions.items():
        rmask = _region_mask(lat1d, lon1d, box['lat'], box['lon'])
        vals = lrp_2d[rmask]
        vals = vals[np.isfinite(vals)]
        out[name] = float(np.nanmean(vals)) if vals.size > 0 else np.nan
    return out


# =============================================================================
# Composite collection (same pattern as cell above)
# =============================================================================

SCENARIO_FLAGS = {'TP': SHOW_TP, 'FP': SHOW_FP, 'TN': SHOW_TN, 'FN': SHOW_FN}
ACTIVE_SCENARIOS = [k for k, v in SCENARIO_FLAGS.items() if v]

for scenario_name in ACTIVE_SCENARIOS:

    sst_per_sr  = []
    lrp_per_sr  = []
    bar_per_sr  = []          # per-region means for each split×seed
    lat_plot    = None
    lon_plot    = None

    for split_idx in range(N_SPLITS):
        res    = results[split_idx]
        sst_tr = res['sst_tr']

        for run_idx in range(N_SEEDS):
            lrp_info = res['lrp_data'][run_idx]
            if lrp_info is None:
                continue

            y_s = res['y_scores_runs'][run_idx]['train']
            y_t = res['y_true']['train']

            prec, rec, thr = precision_recall_curve(y_t, y_s)
            thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
            y_pred = (y_s >= thr_best).astype(int)

            if scenario_name == 'TP':
                mask = (y_pred == 1) & (y_t == 1)
            elif scenario_name == 'FP':
                mask = (y_pred == 1) & (y_t == 0)
            elif scenario_name == 'TN':
                mask = (y_pred == 0) & (y_t == 0)
            else:
                mask = (y_pred == 0) & (y_t == 1)

            lrp_flat = lrp_info['lrp_flat']
            n_lrp    = lrp_flat.shape[0]
            mask_lrp = mask[:n_lrp]
            sst_lrp  = sst_tr[:n_lrp]

            if mask_lrp.sum() == 0:
                continue

            # SST composite (land → NaN)
            sst_m = sst_lrp.astype(float).copy()
            sst_m[sst_m < -5] = np.nan
            sst_mean = np.nanmean(sst_m[mask_lrp], axis=0)

            # Positive-only LRP composite
            lrp_sel  = np.where(lrp_flat[mask_lrp] > 0, lrp_flat[mask_lrp], 0.0)
            lrp_mean = np.mean(lrp_sel, axis=0)

            sst_per_sr.append(sst_mean)
            lrp_per_sr.append(lrp_mean)

            if lat_plot is None:
                lat_plot = res['lat']
                lon_plot = res['lon']

            # Regional mean relevance for this split×seed
            bar_per_sr.append(regional_mean_relevance(lrp_mean, lat_plot, lon_plot, REGIONS))

    if len(sst_per_sr) == 0:
        print(f'{scenario_name}: no samples — skipping.')
        continue

    n_sr      = len(sst_per_sr)
    sst_grand = np.nanmean(np.stack(sst_per_sr, axis=0), axis=0)
    lrp_grand = np.nanmean(np.stack(lrp_per_sr, axis=0), axis=0)

    print(f'{scenario_name}: averaged over {n_sr} split×seed composites')

    lon2d, lat2d = np.meshgrid(lon_plot, lat_plot)

    # ── Normalize ensemble-mean maps (97th percentile) ────────────────
    sst_scale = np.nanpercentile(np.abs(sst_grand), 97)
    sst_grand = np.clip(sst_grand / (sst_scale + 1e-12), -1, 1)

    lrp_scale = np.nanpercentile(lrp_grand, 97)
    lrp_grand = np.clip(lrp_grand / (lrp_scale + 1e-12), 0, 1)

    # ── Percentile-normalised LRP for contour overlay ─────────────────
    pctl = np.nanpercentile(np.abs(lrp_grand), 90) + 1e-12
    lrp_norm = np.clip(lrp_grand / pctl, 0.0, 1.0)
    lrp_smooth = gaussian_filter(lrp_norm, sigma=3)

    # ==================================================================
    # FIGURE 1 — three-panel (a) SST + contour, (b) LRP + boxes, (c) bar
    # ==================================================================
    proj = ccrs.PlateCarree(central_longitude=180)
    fig = plt.figure(figsize=(20, 5.5))
    gs  = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.8], wspace=0.25)

    # ── (a) Mean SST with LRP contour overlay ────────────────────────
    ax_a = fig.add_subplot(gs[0], projection=proj)
    ax_a.set_global()
    ax_a.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax_a.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
    ax_a.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    im_a = ax_a.pcolormesh(
        lon2d, lat2d, sst_grand,
        cmap=cmocean.cm.balance, vmin=-1, vmax=1,
        transform=ccrs.PlateCarree(), shading='auto', zorder=0,
    )
    plt.colorbar(im_a, ax=ax_a, orientation='horizontal', pad=0.04,
                 fraction=0.046, label='SST (normalized)')
    ax_a.set_title('(a)')

    # ── (b) Positive-only LRP with region boxes ──────────────────────
    ax_b = fig.add_subplot(gs[1], projection=proj)
    ax_b.set_global()
    ax_b.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax_b.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
    ax_b.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    im_b = ax_b.pcolormesh(
        lon2d, lat2d, lrp_grand,
        cmap='magma', vmin=0, vmax=1,
        transform=ccrs.PlateCarree(), shading='auto', zorder=0,
    )
    # region boxes
    for name, box in REGIONS.items():
        rect = Rectangle(
            (box['lon'][0], box['lat'][0]),
            box['lon'][1] - box['lon'][0],
            box['lat'][1] - box['lat'][0],
            fill=False, edgecolor=REGION_COLORS[name],
            linewidth=2.5, transform=ccrs.PlateCarree(),
        )
        ax_b.add_patch(rect)
    plt.colorbar(im_b, ax=ax_b, orientation='horizontal', pad=0.04,
                 fraction=0.046, label='Relevance (normalized)')
    ax_b.set_title('(b)')

    # ── (c) Regional bar plot with bootstrap CIs ─────────────────────
    ax_c = fig.add_subplot(gs[2])
    region_names = list(REGIONS.keys())
    n_reg = len(region_names)
    x_pos = np.arange(n_reg)

    # Collect per-region values across split×seeds
    region_vals = {name: [] for name in region_names}
    for bar_dict in bar_per_sr:
        for name in region_names:
            region_vals[name].append(bar_dict[name])

    means, lowers, uppers = [], [], []
    for name in region_names:
        m, lo, hi = bootstrap_mean_ci(region_vals[name], B=1000, random_state=42)
        means.append(m)
        lowers.append(m - lo)
        uppers.append(hi - m)

    bar_colors = [REGION_COLORS[n] for n in region_names]
    ax_c.bar(x_pos, means, yerr=[lowers, uppers],
             color=bar_colors, alpha=0.6, edgecolor='black', capsize=3)
    ax_c.set_xticks(x_pos)
    ax_c.set_xticklabels(region_names, rotation=45, ha='right', fontsize=9)
    ax_c.set_ylabel('Mean positive relevance')
    ax_c.set_title('(c)')

    plt.tight_layout()
    plt.show()

    # ==================================================================
    # FIGURE 2 — pairwise p-values
    # ==================================================================
    pvals = np.full((n_reg, n_reg), np.nan)
    for a in range(n_reg):
        for b in range(a + 1, n_reg):
            pvals[a, b] = bootstrap_p_value_diff_means(
                region_vals[region_names[a]],
                region_vals[region_names[b]],
                B=3000, random_state=42,
            )
            pvals[b, a] = pvals[a, b]

    fig_p, ax_p = plt.subplots(figsize=(8, 6.5))
    im_p = ax_p.imshow(pvals, cmap='RdYlGn_r', vmin=0, vmax=0.10)
    ax_p.set_xticks(range(n_reg))
    ax_p.set_yticks(range(n_reg))
    ax_p.set_xticklabels(region_names, rotation=45, ha='right', fontsize=9)
    ax_p.set_yticklabels(region_names, fontsize=9)

    for a in range(n_reg):
        for b in range(a + 1, n_reg):
            val = pvals[a, b]
            txt = f'{val:.3f}' if np.isfinite(val) else '—'
            ax_p.text(b, a, txt, ha='center', va='center', fontsize=8,
                      color='white' if val < 0.05 else 'black')

    plt.colorbar(im_p, ax=ax_p, label='p-value (permutation)')
    ax_p.set_title('(a)')
    plt.tight_layout()
    plt.show()

In [14]:
mask_valid = np.isfinite(lrp_grand) & np.isfinite(sst_grand)
r = np.corrcoef(
    lrp_grand[mask_valid].ravel(),
    np.abs(sst_grand[mask_valid]).ravel()
)[0, 1]
print(r)

0.7418227837667464
